In [0]:
%sql
-- Students At Risk with Week-over-Week Comparison
-- Shows current count, previous week count, and percentage change

WITH weekly_counts AS (
  SELECT 
    DATE_TRUNC('week', _gold_timestamp) as week_start,
    COUNT(*) as students_at_risk
  FROM workspace.default.gold_at_risk_students
  WHERE _gold_timestamp >= CURRENT_DATE - INTERVAL '30' DAY
  GROUP BY DATE_TRUNC('week', _gold_timestamp)
),
lagged_counts AS (
  SELECT
    week_start,
    students_at_risk as current_count,
    LAG(students_at_risk) OVER (ORDER BY week_start) as previous_count
  FROM weekly_counts
)
SELECT
  week_start,
  current_count,
  previous_count,
  current_count - previous_count as change_count,
  ROUND((current_count - previous_count) * 100.0 / NULLIF(previous_count, 0), 2) as pct_change,
  CASE 
    WHEN current_count > previous_count THEN '📈 Increasing'
    WHEN current_count < previous_count THEN '📉 Decreasing'
    ELSE '➡️ Stable'
  END as trend_direction
FROM lagged_counts
WHERE previous_count IS NOT NULL
ORDER BY week_start DESC
LIMIT 1

In [0]:
%sql
-- High Risk Students with 7-Day Moving Average
-- Tracks immediate intervention cases with trend smoothing

WITH daily_high_risk AS (
  SELECT 
    DATE(_gold_timestamp) as risk_date,
    COUNT(*) as high_risk_count
  FROM workspace.default.gold_at_risk_students
  WHERE intervention_tier = 'HIGH'
    AND _gold_timestamp >= CURRENT_DATE - INTERVAL '30' DAY
  GROUP BY DATE(_gold_timestamp)
),
moving_avg AS (
  SELECT
    risk_date,
    high_risk_count,
    ROUND(AVG(high_risk_count) OVER (
      ORDER BY risk_date
      ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    ), 2) as moving_avg_7d
  FROM daily_high_risk
)
SELECT
  risk_date,
  high_risk_count as current_high_risk,
  moving_avg_7d,
  ROUND((high_risk_count - moving_avg_7d) * 100.0 / NULLIF(moving_avg_7d, 0), 2) as deviation_pct,
  CASE
    WHEN high_risk_count > moving_avg_7d * 1.2 THEN '🚨 Spike Detected'
    WHEN high_risk_count < moving_avg_7d * 0.8 THEN '✅ Improving'
    ELSE '➡️ Normal'
  END as status
FROM moving_avg
ORDER BY risk_date DESC
LIMIT 14

In [0]:
%sql
-- Risk Score Moving Average with Daily Trends
-- Provides overall risk level indicator across all students

WITH daily_risk_scores AS (
  SELECT 
    DATE(_gold_timestamp) as score_date,
    AVG(risk_score) as avg_risk_score,
    MAX(risk_score) as max_risk_score,
    MIN(risk_score) as min_risk_score,
    STDDEV(risk_score) as stddev_risk_score,
    COUNT(*) as student_count
  FROM workspace.default.gold_at_risk_students
  WHERE _gold_timestamp >= CURRENT_DATE - INTERVAL '30' DAY
  GROUP BY DATE(_gold_timestamp)
)
SELECT
  score_date,
  ROUND(avg_risk_score, 4) as daily_avg_risk,
  ROUND(max_risk_score, 4) as daily_max_risk,
  ROUND(min_risk_score, 4) as daily_min_risk,
  ROUND(stddev_risk_score, 4) as risk_volatility,
  student_count,
  ROUND(AVG(avg_risk_score) OVER (
    ORDER BY score_date
    ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
  ), 4) as moving_avg_7d,
  CASE
    WHEN avg_risk_score >= 0.75 THEN '🔴 Critical'
    WHEN avg_risk_score >= 0.50 THEN '🟡 Elevated'
    ELSE '🟢 Moderate'
  END as risk_level
FROM daily_risk_scores
ORDER BY score_date DESC